In [1]:
# chain通常将大型语言模型LLM与提示词Prompt结合在一起
# 加载环境变量
import os

api_key = os.environ.get("DEEPSEEK_API_KEY")

In [6]:
import pandas as pd

df = pd.read_csv("Data.csv")
df

,product,review,rating
0,Leaf Blower,This leaf blower is amazing! It has great powe...,5
1,Leaf Blower,I'm very satisfied with this product. It's pow...,4
2,Leaf Blower,Good value for money. Works well for small yards.,3
3,Garden Hose,This garden hose is durable and doesn't kink. ...,5
4,Garden Hose,The hose is okay but the connections leak a bi...,3
5,Lawn Mower,Excellent lawn mower! Cuts grass evenly and is...,5
6,Lawn Mower,Works well but is a bit noisy. Still does a go...,4
7,Pruning Shears,These pruning shears are sharp and comfortable...,5
8,Pruning Shears,Good quality shears. They stay sharp for a lon...,4
9,Garden Gloves,These garden gloves are comfortable and protec...,5


In [8]:
df.head()

,product,review,rating
0,Leaf Blower,This leaf blower is amazing! It has great powe...,5
1,Leaf Blower,I'm very satisfied with this product. It's pow...,4
2,Leaf Blower,Good value for money. Works well for small yards.,3
3,Garden Hose,This garden hose is durable and doesn't kink. ...,5
4,Garden Hose,The hose is okay but the connections leak a bi...,3


In [9]:
# LLM Chain

In [13]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [14]:
llm = ChatOpenAI(
    base_url="https://api.deepseek.com/",
    api_key=api_key,
    model="deepseek-v4-flash",
    temperature=0.9
    # 使用较高的值来初始化ChatOpenAI
)

In [15]:
# 初始化一个提示词模板
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

In [18]:
# 将二者合为一条链
chain = LLMChain(llm = llm , prompt = prompt)

C:\Users\hilary\AppData\Local\Temp\ipykernel_19372\3281214333.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm = llm , prompt = prompt)


In [20]:
product = "Queen Size Sheet Set"
chain.run(product)

'A company that makes queen size sheet sets is best described as a:\n\n**Bedding Manufacturer**  \nor more specifically a **Home Textiles Company / Linens Manufacturer**.\n\nIf you’re looking for a *brand name* to describe it, here are some options:\n\n- Queen & Quilt  \n- The Sheet Co.  \n- Sweet Queen Bedding  \n- Royal Rest Linens  \n- PureThread Sheets  \n- Queen Size Comfort  \n- The Bedding Atelier  \n- SleepHaven Textiles  \n\nBest descriptive name depends on tone:  \n- **Professional/industrial:** “Queen Size Bedding Co.”  \n- **Elegant/retail:** “Linen & Queen”  \n- **Modern/online:** “Queensheet” or “SheetQueen”'

In [22]:
# 顺序链(Sequential Chains)
from langchain_classic.chains import SimpleSequentialChain

In [24]:
# prompt _template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm,prompt=first_prompt)

In [30]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following company: {company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [31]:
overall_simple_chain = SimpleSequentialChain(
    chains=[chain_one,chain_two],
    verbose=True
)

In [33]:
overall_simple_chain.run(product) # 简单顺序链在只有一个输入和输出时表现非常优秀



> Entering new SimpleSequentialChain chain...
The best **descriptive** name for a company that makes Queen Size Sheet Sets is:

**“Bedding Manufacturer”** or **“Home Textiles Company”**

If you want to be more specific, you could say:

- **Queen-Size Sheet Set Manufacturer**
- **Linen & Bedding Producer**
- **Sleep Products Company**

If you’re looking for a **brand name** instead, something like:

- **QueenRest Linens**
- **SheetCraft Co.**
- **RoyalSleep Bedding**

But generally, the clearest and most accurate industry term is **bedding manufacturer**.
A bedding manufacturer specializing in queen-size sheet sets, producing high-quality home textiles and sleep products under trusted, well-known brand names.

> Finished chain.


'A bedding manufacturer specializing in queen-size sheet sets, producing high-quality home textiles and sleep products under trusted, well-known brand names.'

In [43]:
# 使用SequentialChain来处理多输出输入
from langchain_classic.chains import SequentialChain

first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to Chinese:"
    "\n\n{Review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, output_key="Chinese_Review")

In [42]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you Summarize the following review in 1 sentence:"
    "\n\n{Chinese_Review}"
)
chain_two = LLMChain(llm=llm, prompt=second_prompt, output_key="summary")

In [41]:
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
chain_three = LLMChain(llm=llm, prompt=third_prompt, output_key="language")

In [111]:
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nlanguage: {language}"
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt, output_key="followingup_message")

In [50]:
overall_chain = SequentialChain(
    chains=[
        chain_one,
        chain_two,
        chain_three,
        chain_four
    ],
    input_variables=["Review"],
    output_variables=["Chinese_Review","summary","language","followingup_message"],
    verbose=True
)

In [51]:
review = df.review[5] # 取出第6行数据
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'Review': 'Excellent lawn mower! Cuts grass evenly and is easy to maneuver.',
 'Chinese_Review': '优秀的割草机！割草均匀，操控轻松。',
 'summary': '这款割草机表现出色，割草均匀且操控轻松。',
 'language': 'English',
 'followingup_message': '这款割草机表现出色，割草均匀且操控轻松。'}

In [53]:
# 路由链，存在多条子链，每条子链处理某种特定类型的输入
# 首先判断使用哪条子链，然后将输入传递到相应的子链
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are very good at math reasoning, with a formal but helpful tone. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to a comprehensive answer.

Here is a question:
{input}"""


history_template = """You are a very good historian. \
You have a deep understanding of historical contexts from a range of historical periods. \
You have the ability to think, reflect, and evaluate the past. \
You have a respect for historical evidence and the ability to make use of it to support \
your explanations and judgements.

Here is a question:
{input}"""


computerscience_template = """
You are a successful computer scientist. \
You have a passion for forward-thinking, confidence, strong problem-solving capabilities, \
and excellent communication skills. \
You are great at answering coding questions. \
You are so good because you know how to solve a problem by first understanding the problem, \
that a machine can easily interpret and you know how to choose the optimal algorithm for a \
time complexity and space complexity.

Here is a question:
{input}"""


In [54]:
# 针对不同的类型创建提示词模板
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {   "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    },
]

In [89]:
from langchain_classic.chains.router import MultiPromptChain # 用于在多个模板提示词之间路由
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser # 借助LLN帮助在不同的子链之间路由
# 解析器可以将大模型输出解析成一个字典，根据字典的内容可以在下游确定使用哪一条链，以及确定该条链的输入是什么
from langchain_core.prompts import PromptTemplate

In [56]:
# 重新初始化 LLM: 这里 temperature=0(前面 Cell 5 用的是 0.9)
# 路由链需要大模型稳定、按格式输出 JSON 路由结果,
# 温度太高会导致 JSON 格式错乱或选链随机, 所以这里要低随机性
llm = ChatOpenAI(
    base_url="https://api.deepseek.com/",
    api_key=api_key,
    model="deepseek-v4-flash",
    temperature=0
)

In [109]:
# ========== 为每个学科创建"目标链"(destination chains) ==========
# 遍历 Cell 21 的 prompt_infos, 把每个学科的模板和 LLM 组装成一条 LLMChain
# destination_chains 是个字典: {"physics": 对应的链, "math": 对应的链, ...}
# 之后路由链判断出问题属于哪个学科, 就按名字从这里取对应的链来回答
destination_chains  = {}
for p_info in prompt_infos:
    name = p_info["name"]                        # 子链的名字, 如 "physics"
    prompt_template = p_info["prompt_template"]  # 该学科专属的提示词模板
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)     # 模板 + LLM = 一条子链
    destination_chains[name] = chain             # 按名字登记进字典

# 把四个子链拼成 "名字:描述" 的清单, 后面要作为候选名单喂给路由链
# 例如其中一条: "physics:Good for answering questions about physics"
destinations = [f"{p['name']}:{p['description']}" for p in prompt_infos]
# 用换行符把清单里的每一项连接成一个完整字符串(填路由模板的 {destinations} 用)
destination_str = "\n".join(destinations)
print(destination_str)

physics:Good for answering questions about physics
math:Good for answering math questions
History:Good for answering history questions
computer science:Good for answering computer science questions


In [100]:
# 默认链: 当问题不属于任何学科(路由器选了 DEFAULT)时使用
# 模板里只有 {input}, 相当于把用户问题原样交给大模型, 充当通用问答
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [66]:
# 路由器的提示词模板: 让大模型扮演"调度员"
# 它不回答问题, 只负责从候选清单里挑出最适合处理该问题的子链名
# 关键段落:
#   << FORMATTING >>         规定大模型必须按 json 格式返回(destination + next_inputs)
#   << CANDIDATE PROMPTS >>  候选子链清单, 下个 Cell 会用 destination_str 填入 {destinations}
#   << INPUT >>              用户原始问题, 填入 {input}
# 说明: 模板里写成 {{{{ 和 {{ 是转义, 经过 .format() 后会还原成 {,
#      这样大模型最终看到的 JSON 示例才是正常的单花括号
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a language model select the model prompt best suited for the input. \
You will be given the names of the available prompt templates and a description of what the prompt template is best suited for. \
You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \\ name of the prompt to use or "DEFAULT"
    "next_inputs": string \\ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt names specified below unless it can be best answered by "DEFAULT".
REMEMBER: "next_inputs" can just be the original input if you don't need to modify it.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""


In [91]:
# 第一步: 把候选清单填进路由模板的 {destinations} 占位符
# 填完后模板里只剩 {{input}} 一个待填项
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destination_str
)

# 第二步: 组装路由链的提示词
#   input_variables=["input"]           输入是用户原始问题
#   output_parser=RouterOutputParser()  把大模型返回的 ```json``` 代码块
#       解析成字典, 如 {"destination": "physics", "next_inputs": "原问题"}
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)

# 第三步: 生成路由链本体(只做"选链"这一件事, 不负责回答)
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [103]:
# 组装最终的路由链: 调度员(router_chain) + 各学科子链(destination_chains) + 兜底(default_chain)
# 必须用变量 chain 接住返回对象, 后面 chain.run() 调用的就是它
# 运行流程: chain.run(问题)
#   → router_chain 判断问题属于哪个学科
#   → 命中: 从 destination_chains 取对应子链来回答
#   → 未命中: 走 default_chain, 作为通用模型回答
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True
)

MultiPromptChain(verbose=True, router_chain=LLMRouterChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['input'], input_types={}, output_parser=RouterOutputParser(), partial_variables={}, template='Given a raw text input to a language model select the model prompt best suited for the input. You will be given the names of the available prompt templates and a description of what the prompt template is best suited for. You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.\n\n<< FORMATTING >>\nReturn a markdown code snippet with a JSON object formatted to look like:\n```json\n{{\n    "destination": string \\ name of the prompt to use or "DEFAULT"\n    "next_inputs": string \\ a potentially modified version of the original input\n}}\n```\n\nREMEMBER: "destination" MUST be one of the candidate prompt names specified below unless it can be best answered by "DEFAULT".\nR

In [105]:
# 物理问题 → 应被路由到 physics 子链, 以"物理教授"人设回答(verbose 输出可见路由过程)
chain.run("What is black body radiation,response in Chinese")

'**黑体辐射**（Blackbody Radiation）是指一个理想化的物体——**黑体** 所发出的电磁辐射。\n\n黑体是一个理想模型，它能够**吸收所有入射到其表面的电磁波**，不会反射或透射任何辐射。因为吸收率是 100%，所以当它处于热平衡状态时，它会以特定的光谱向外辐射能量。这种辐射只取决于黑体的**温度**，与材料、形状等无关。\n\n---\n\n### 核心特征\n\n1. **连续光谱**  \n   黑体辐射是连续的，覆盖从红外到可见光再到紫外的所有波长。\n\n2. **温度决定一切**  \n   - 温度越高，辐射的总能量越大。\n   - 温度越高，辐射的峰值波长越短，即光谱整体向短波方向移动。\n\n3. **普朗克公式**  \n   德国物理学家马克斯·普朗克在 1900 年提出了黑体辐射的能量分布公式：\n\n   \\[\n   B(\\lambda, T) = \\frac{2hc^2}{\\lambda^5} \\cdot \\frac{1}{e^{\\frac{hc}{\\lambda k_B T}} - 1}\n   \\]\n\n   其中：\n   - \\(B\\) 是光谱辐射亮度；\n   - \\(h\\) 是普朗克常数；\n   - \\(c\\) 是光速；\n   - \\(k_B\\) 是玻尔兹曼常数；\n   - \\(\\lambda\\) 是波长；\n   - \\(T\\) 是绝对温度。\n\n4. **维恩位移定律**  \n   辐射峰值波长与温度成反比：\n\n   \\[\n   \\lambda_{\\max} \\approx \\frac{2.898 \\times 10^{-3}}{T} \\ \\text{m}\n   \\]\n\n5. **斯特藩-玻尔兹曼定律**  \n   单位面积辐射的总功率与温度的四次方成正比：\n\n   \\[\n   P = \\sigma T^4\n   \\]\n\n   其中 \\(\\sigma\\) 是斯特藩-玻尔兹曼常数。\n\n---\n\n### 物理意义与历史\n\n黑体辐射是量子力学诞生的关键线索。经典物理学（瑞利-金斯公式）在短波区域预测能量无限增大，这被称为“紫外灾难”。普朗克为解决这个问题，首次提出了**能量量子化

In [106]:
# 数学问题 → 应被路由到 math 子链
chain.run("什么是微积分")

'微积分（Calculus）是数学中研究**变化**与**积累**的核心分支。它主要由两大部分组成：\n\n1. **微分学**  \n   研究“瞬间变化率”，也就是**导数**。  \n   比如：物体在某一时刻的速度、函数在某一点的变化趋势。  \n   它解决的是“切线的斜率”“瞬时速度”“最优化”等问题。\n\n2. **积分学**  \n   研究“连续量的累积”，也就是**积分**。  \n   比如：速度曲线下的面积就是路程；密度分布的总质量。  \n   它解决的是“面积”“体积”“总量”等问题。\n\n微积分最重要的桥梁是**微积分基本定理**：  \n> 微分和积分是互逆运算。  \n> 求导之后再积分，或者积分之后再求导，会回到原来的函数（相差一个常数）。\n\n简单地说：\n\n- **微分**把一个整体拆成无穷小的局部来看变化；\n- **积分**把无穷小的局部累加起来得到整体。\n\n在计算机科学中，微积分也非常有用：机器学习中的梯度下降依赖导数，图像处理中的边缘检测依赖离散微分，概率论中的连续分布依赖积分，物理引擎中的运动模拟依赖常微分方程求解。\n\n所以，微积分不只是“高数课上的公式”，而是一套理解**动态世界**的语言。'

In [108]:
# 生物问题, 不在四个学科里 → 路由器应选 DEFAULT, 走默认链做通用回答
chain.run("Why does every cell in our body contain DNA?")

'Every cell in our body contains DNA because DNA is the **master blueprint** for building and maintaining that cell.\n\nHere’s the core logic:\n\n1. **All of your cells come from one fertilized egg.**\n   That single zygote contained your full genome—a complete set of instructions encoded in DNA.\n\n2. **When cells divide, they copy and pass on that DNA.**\n   This ensures each daughter cell inherits the same instructions. So nearly every cell in your body has the same genetic code.\n\n3. **DNA stores the instructions for making proteins.**\n   Proteins do almost everything in the body: they build structures, catalyze reactions, transport molecules, signal between cells, and more. DNA doesn’t do the work directly—it provides the code that tells the cell which proteins to make and when.\n\n4. **Cells need those instructions throughout their life.**\n   Even after a cell has differentiated into, say, a muscle cell or a nerve cell, it still needs to continually read genes to produce prote